In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv'

In [3]:
!wget $data -O data-lead-scoring.csv 

--2025-10-14 07:37:30--  https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 80876 (79K) [text/plain]
Saving to: ‘data-lead-scoring.csv’

data-lead-scoring.c 100%[===================>]  78.98K   387KB/s    in 0.2s    

2025-10-14 07:37:31 (387 KB/s) - ‘data-lead-scoring.csv’ saved [80876/80876]



In [47]:
df = pd.read_csv('data-lead-scoring.csv')
df.head().T

,0,1,2,3,4
lead_source,paid_ads,social_media,events,paid_ads,referral
industry,NaN,retail,healthcare,retail,education
number_of_courses_viewed,1,1,5,2,3
annual_income,79450.0,46992.0,78796.0,83843.0,85012.0
employment_status,unemployed,employed,unemployed,NaN,self_employed
location,south_america,south_america,australia,australia,europe
interaction_count,4,1,3,1,3
lead_score,0.94,0.8,0.69,0.87,0.62
converted,1,0,1,0,1


In [48]:
df.dtypes

lead_source                  object
industry                     object
number_of_courses_viewed      int64
annual_income               float64
employment_status            object
location                     object
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [49]:
df.isnull().sum()

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

In [50]:
df.columns

Index(['lead_source', 'industry', 'number_of_courses_viewed', 'annual_income',
       'employment_status', 'location', 'interaction_count', 'lead_score',
       'converted'],
      dtype='object')

In [51]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']

In [52]:
numerical = ['number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score']

In [53]:
df[categorical] = df[categorical].fillna('NA')

In [54]:
df[numerical] = df[numerical].fillna(0.0)

In [55]:
df.isnull().sum()

lead_source                 0
industry                    0
number_of_courses_viewed    0
annual_income               0
employment_status           0
location                    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [56]:
df.industry.value_counts()

industry
retail           203
finance          200
other            198
healthcare       187
education        187
technology       179
manufacturing    174
NA               134
Name: count, dtype: int64

#### Q1. Mode for industry column

In [57]:
df.industry.mode()

0    retail
Name: industry, dtype: object

#### Q2. Correllation matrix

In [58]:
df[numerical].corr()

,number_of_courses_viewed,annual_income,interaction_count,lead_score
number_of_courses_viewed,1.000000,0.009770,-0.023565,-0.004879
annual_income,0.009770,1.000000,0.027036,0.015610
interaction_count,-0.023565,0.027036,1.000000,0.009888
lead_score,-0.004879,0.015610,0.009888,1.000000


#### Data split

In [59]:
from sklearn.model_selection import train_test_split

In [60]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [61]:
len(df_train), len(df_val), len(df_test)

(876, 293, 293)

In [62]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [63]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [64]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

#### Q3. Mutual Info score

In [65]:
from sklearn.metrics import mutual_info_score

In [68]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train.converted)

In [69]:
mi = df_full_train[categorical].apply(mutual_info_converted_score)
mi.sort_values(ascending=False)

lead_source          0.025665
employment_status    0.013258
industry             0.011685
location             0.002253
dtype: float64

#### Q4. Model Training

In [70]:
from sklearn.feature_extraction import DictVectorizer

In [71]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [72]:
from sklearn.linear_model import LogisticRegression

In [73]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [74]:
y_pred = model.predict_proba(X_val)[:, 1]

In [75]:
conversion_decision = (y_pred >= 0.5)

In [81]:
acc = (y_val == conversion_decision).mean().round(2)
acc

np.float64(0.7)

In [94]:
def train_logistic_regression(c_val, features):
    dv = DictVectorizer(sparse=False)
    model = LogisticRegression(solver='liblinear', C=c_val, max_iter=1000, random_state=42)
    
    train_dict = df_train[features].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)

    val_dict = df_val[features].to_dict(orient='records')
    X_val = dv.transform(val_dict)
    
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    conversion_decision = (y_pred >= 0.5)
    
    return (y_val == conversion_decision).mean()

#### Q5. Feature elimination

In [105]:
# model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
combined_features = categorical + numerical
for f in combined_features:
    features = combined_features.copy()
    features.remove(f)
    # filtered_features = combined_features.filter(lambda x: x == f, combined_features)
    new_acc = train_logistic_regression(1.0, features)
    print(f, abs(acc - new_acc))
    

lead_source 0.0030716723549488734
industry 0.0003412969283276279
employment_status 0.003754266211604018
location 0.009897610921501765
number_of_courses_viewed 0.14368600682593857
annual_income 0.1532423208191127
interaction_count 0.14368600682593857
lead_score 0.006484641638225264


#### Q6. Regularization

In [108]:
c = [0.01, 0.1, 1, 10, 100]
for c_val in c:
    reg_acc = train_logistic_regression(c_val, categorical + numerical)
    print(c_val,':', reg_acc)

0.01 : 0.6996587030716723
0.1 : 0.6996587030716723
1 : 0.6996587030716723
10 : 0.6996587030716723
100 : 0.6996587030716723
